In [0]:
-- ============================================================
-- AEGIS — ANALYTICAL TOOL LAYER
-- ============================================================
-- Purpose:
-- Controlled, parameterized analytical interfaces for the
-- future Aegis agent.
--
-- The agent should query these functions rather than directly
-- manipulating raw Silver tables.
--
-- All calculations remain deterministic SQL.
-- GenAI is responsible for intent, tool selection and synthesis.
-- ============================================================


-- ============================================================
-- 1. SUPPLIER PERFORMANCE TOOL
-- ============================================================
-- Example:
-- SELECT *
-- FROM workspace.aegis_gold.get_supplier_performance(
--     'SUP0014',
--     DATE('2025-01-01'),
--     DATE('2025-04-30')
-- );

CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_supplier_performance(
    p_supplier_id STRING,
    p_start_date DATE,
    p_end_date DATE
)
RETURNS TABLE
RETURN
SELECT
    month,
    supplier_id,
    supplier_name,
    region,
    supplier_type,
    reliability_score,
    lead_time_variability,
    orders,
    units_supplied,
    revenue,
    shipments,
    avg_actual_transit_days,
    avg_delivery_delay_days,
    late_delivery_rate,
    product_count,
    avg_agreed_lead_time
FROM workspace.aegis_gold.supplier_performance
WHERE supplier_id = p_supplier_id
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;


-- ============================================================
-- 2. DELIVERY PERFORMANCE TOOL
-- ============================================================
-- Can investigate:
--   warehouse
--   carrier
--   time period
--
-- Example:
-- SELECT *
-- FROM workspace.aegis_gold.get_delivery_performance(
--     'WH0003',
--     NULL,
--     DATE('2024-11-01'),
--     DATE('2025-01-31')
-- );

CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_delivery_performance(
    p_warehouse_id STRING,
    p_carrier_id STRING,
    p_start_date DATE,
    p_end_date DATE
)
RETURNS TABLE
RETURN
SELECT
    month,
    warehouse_id,
    warehouse_region,
    carrier_id,
    carrier_name,
    service_type,
    shipment_count,
    late_shipments,
    late_delivery_rate,
    avg_actual_transit_days,
    avg_expected_transit_days,
    avg_delivery_delay_days,
    max_delivery_delay_days
FROM workspace.aegis_gold.delivery_performance
WHERE (p_warehouse_id IS NULL OR warehouse_id = p_warehouse_id)
  AND (p_carrier_id IS NULL OR carrier_id = p_carrier_id)
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;


-- ============================================================
-- 3. INVENTORY HEALTH TOOL
-- ============================================================
-- Designed for:
--   product investigation
--   warehouse investigation
--   stockout analysis
--
-- Example:
-- SELECT *
-- FROM workspace.aegis_gold.get_inventory_health(
--     'WH0003',
--     'PROD000123',
--     DATE('2025-02-01'),
--     DATE('2025-02-28')
-- );


CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_inventory_health(
    p_warehouse_id STRING,
    p_product_id STRING,
    p_start_date DATE,
    p_end_date DATE
)
RETURNS TABLE
RETURN
SELECT
    snapshot_date,
    warehouse_id,
    product_id,
    product_name,
    category,
    warehouse_region,
    warehouse_capacity,
    stock_quantity,
    reserved_quantity,
    available_quantity,
    stockout_flag,
    stockout_indicator,
    demand_index,
    inventory_demand_ratio
FROM workspace.aegis_gold.inventory_health
WHERE (p_warehouse_id IS NULL OR warehouse_id = p_warehouse_id)
  AND (p_product_id IS NULL OR product_id = p_product_id)
  AND snapshot_date >= p_start_date
  AND snapshot_date <= p_end_date
ORDER BY snapshot_date;


-- ============================================================
-- 4. WAREHOUSE PERFORMANCE TOOL
-- ============================================================
-- Example:
-- SELECT *
-- FROM workspace.aegis_gold.get_warehouse_performance(
--     'WH0003',
--     DATE('2024-11-01'),
--     DATE('2025-01-31')
-- );


CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_warehouse_performance(
    p_warehouse_id STRING,
    p_start_date DATE,
    p_end_date DATE
)
RETURNS TABLE
RETURN
SELECT
    month,
    warehouse_id,
    region,
    warehouse_type,
    capacity,
    processing_efficiency,
    shipments,
    late_shipments,
    late_delivery_rate,
    avg_actual_transit_days,
    avg_delivery_delay_days,
    avg_stock_quantity,
    avg_available_quantity,
    stockout_rate
FROM workspace.aegis_gold.warehouse_performance
WHERE warehouse_id = p_warehouse_id
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;


-- ============================================================
-- 5. CARRIER PERFORMANCE TOOL
-- ============================================================
-- Example:
-- SELECT *
-- FROM workspace.aegis_gold.get_carrier_performance(
--     'CAR0007',
--     DATE('2025-05-01'),
--     DATE('2025-07-31')
-- );


CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_carrier_performance(
    p_carrier_id STRING,
    p_start_date DATE,
    p_end_date DATE
)
RETURNS TABLE
RETURN
SELECT
    month,
    carrier_id,
    carrier_name,
    service_type,
    base_transit_days,
    transit_variability,
    shipment_count,
    late_shipments,
    late_delivery_rate,
    avg_actual_transit_days,
    avg_delivery_delay_days,
    transit_time_stddev
FROM workspace.aegis_gold.carrier_performance
WHERE carrier_id = p_carrier_id
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;


-- ============================================================
-- 6. SUPPLY CHAIN RISK TOOL
-- ============================================================
-- This exposes transparent deterministic risk signals.
--
-- IMPORTANT:
-- No ground-truth incident information is used here.
--
-- Example:
-- SELECT *
-- FROM workspace.aegis_gold.get_supply_chain_risk(
--     'SUP0014',
--     DATE('2025-01-01'),
--     DATE('2025-04-30')
-- );


CREATE OR REPLACE FUNCTION
workspace.aegis_gold.get_supply_chain_risk(
    p_supplier_id STRING,
    p_start_date DATE,
    p_end_date DATE
)
RETURNS TABLE
RETURN
SELECT
    month,
    supplier_id,
    reliability_score,
    lead_time_variability,
    late_delivery_rate,
    avg_delivery_delay_days,
    avg_actual_transit_days,
    avg_agreed_lead_time,
    stockout_rate,
    delivery_risk_signal,
    delay_risk_signal,
    inventory_risk_signal,
    supplier_reliability_signal,
    risk_signal_score
FROM workspace.aegis_gold.supply_chain_risk
WHERE supplier_id = p_supplier_id
  AND month >= DATE_TRUNC('month', p_start_date)
  AND month <= DATE_TRUNC('month', p_end_date)
ORDER BY month;

